# 🧠 Google Gemini API (google-generativeai) Kapsamlı Python Rehberi

Bu notebook, Google'ın gelişmiş büyük dil modelleri ailesi olan **Gemini**'ye Python programlama dili üzerinden erişmeyi ve bu modelleri projelerimizde kullanmayı öğreten kapsamlı bir kılavuzdur.

> **Not:** Gemini API'yi kullanabilmek için `google-generativeai` kütüphanesi yüklü olmalıdır. Yükleme komutu: `pip install google-generativeai`.

## 1. Gemini API Kurulumu ve Kimlik Doğrulama

Gemini API'sini kullanabilmek için öncelikle **Google AI Studio** üzerinden bir API anahtarı (API Key) edinmeniz gerekir. Aldığınız anahtarı Python ortamında yapılandırarak istek göndermeye başlayabilirsiniz.

### 1.1. API Yapılandırması
```python
import google.generativeai as genai

# API Anahtarınızı buraya girin
API_KEY = "AIzaSy..."
genai.configure(api_key=API_KEY)
```

### 1.2. Güvenlik İpucu: Çevre Değişkeni Kullanımı
API anahtarını kod içerisine açık metin olarak yazmak büyük bir güvenlik riskidir (özellikle GitHub'a yüklenen projelerde). En iyi yöntem anahtarı bir çevre değişkeni (Environment Variable) olarak saklamaktır:
```python
import os
import google.generativeai as genai

# Sistemden GEMINI_API_KEY çevre değişkenini okur ve otomatik yapılandırır
genai.configure(api_key=os.environ.get("GEMINI_API_KEY"))
```

## 2. Kullanılabilir Modelleri Listeleme

Google Gemini sürekli güncellenen ve yeni modeller barındıran bir ekosistemdir. `list_models()` metodu ile API anahtarınızla erişebileceğiniz modelleri sorgulayabilirsiniz.

```python
# generateContent metodunu destekleyen modelleri filtreleme
for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(f"Model Adı: {model.name} | Açıklama: {model.description}")
```

### Sık Kullanılan Modeller:
- **`gemini-2.0-flash`**: En yeni, hızlı ve yüksek yetenekli varsayılan model.
- **`gemini-1.5-flash`**: Yüksek hız ve düşük maliyet gerektiren genel işler için ideal.
- **`gemini-1.5-pro`**: Karmaşık akıl yürütme (reasoning), kod yazma ve çok modlu (multimodal) büyük dosyalar için tasarlanmış model.

## 3. Üretim Yapılandırması (Generation Config)

Modelin yanıt üretirken uyması gereken kuralları, yaratıcılık seviyesini ve token sınırlarını belirleyebilirsiniz.

| Parametre | Açıklama | Örnek Değer |
|---|---|---|
| **`temperature`** | Çıktının rastgeleliğini belirler. 0'a yakın değerler daha net ve tutarlı, 1'e yakın değerler daha yaratıcıdır. | `0.7` |
| **`top_p`** | Çekirdek örnekleme (nucleus sampling) olasılığı havuzu. | `0.95` |
| **`top_k`** | Seçilebilecek en olası kelime sayısı sınırı. | `40` |
| **`max_output_tokens`** | Üretilecek maksimum cevap uzunluğu (token). | `2048` |
| **`response_mime_type`** | Yanıt formatı. Düz metin için `text/plain`, yapılandırılmış JSON için `application/json` tercih edilebilir. | `"text/plain"` |

```python
generation_config = {
    "temperature": 0.4,
    "top_p": 0.9,
    "max_output_tokens": 1024,
    "response_mime_type": "text/plain"
}
```

## 4. İçerik Üretme ve Sistem Talimatları (System Instructions)

### 4.1. Tek Seferlik İstek (Generate Content)
Karşılıklı diyalog gerektirmeyen, soru-cevap veya özetleme gibi işlemler için kullanılır:
```python
model = genai.GenerativeModel("gemini-2.0-flash")
response = model.generate_content("Python'da liste kavrayışı (list comprehension) nedir?")
print(response.text)
```

### 4.2. Sistem Talimatı (System Instruction) Ekleme
Modele bir rol veya kişilik biçmek (örneğin kurumsal dijital ikiz yapmak) için kullanılır:
```python
model = genai.GenerativeModel(
    model_name="gemini-2.0-flash",
    system_instruction="Sen deneyimli bir yazılım eğitmenisin. Yanıtlarında sadece Türkçe dilini kullan ve örnek kodlar paylaş."
)
```

## 5. Sohbet Oturumları ve Geçmiş Yönetimi

Diyalog tabanlı uygulamalarda (chatbot), konuşmanın geçmişini akılda tutmak için `start_chat` yapısı kullanılır. Bu yapı, gönderilen yeni mesajla birlikte tüm sohbet geçmişini otomatik olarak API'ye paketleyip iletir.

```python
# Örnek bir konuşma geçmişi (model-user formatında)
history = [
    {"role": "user", "parts": ["Merhaba, bugün ne hakkında konuşacağız?"]},
    {"role": "model", "parts": ["Merhaba! İstediğin herhangi bir yazılım konusu hakkında konuşabiliriz. Hazırım! 😊"]}
]

chat = model.start_chat(history=history)
response = chat.send_message("Bana Flask kütüphanesini kısaca açıklar mısın?")
print(response.text)
```

## 6. Hata Yönetimi ve Kota Limitleri (Rate Limits)

Gemini API'sini kullanırken en sık karşılaşılan hatalar ve çözümleri:

1. **`429 Too Many Requests (Quota Exceeded)`**:
   - **Neden:** Ücretsiz pakette dakika başına veya günlük istek sınırını aştınız.
   - **Çözüm:** İstekler arasına `time.sleep()` ile bekleme ekleyin veya ücretli pakete (pay-as-you-go) geçin.
   
2. **`404 Model Not Found`**:
   - **Neden:** Seçtiğiniz model adı yanlış veya API sürümüyle uyumsuz.
   - **Çözüm:** `genai.list_models()` ile güncel listeyi çekin veya model adının başına `models/` eklenip eklenmediğini kontrol edin.

3. **`Blocked Prompt (Safety Filters)`**:
   - **Neden:** Gönderdiğiniz metin Gemini'ın güvenlik filtrelerine takıldı.
   - **Çözüm:** Güvenlik yapılandırmalarını (`safety_settings`) gevşetin veya sorgunuzu yeniden düzenleyin.

## 7. Uygulamalı Kod Bloğu (Gemini API Testi)

Aşağıdaki kod bloğunda, API anahtarınızı kullanarak temel bir generate content sorgusunu çalıştırabilirsiniz.

In [ ]:
import os
import google.generativeai as genai

api_key = os.environ.get("GEMINI_API_KEY", "")

if not api_key:
    print("Lütfen test için GEMINI_API_KEY çevre değişkenini bilgisayarınıza tanımlayın!")
else:
    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel("gemini-2.0-flash")
        
        print("Gemini API'ye istek gönderiliyor...")
        response = model.generate_content("Gemini API nedir? 1 cümlede açıkla.")
        
        print("\n--- Model Yanıtı ---")
        print(response.text)
        
    except Exception as e:
        print(f"İstek gönderilirken hata oluştu: {e}")